# 02 · Silver data quality checks
Quality gate between silver and gold. **Critical** checks (empty tables, null or duplicate keys, unparsed values, invalid amounts) stop the pipeline. **Warnings** (orphan keys, small calculation differences) are reported but don't block it.

Each run's results are appended to `silver.dq_check_results` for auditing.

In [0]:
%run ../../config/pipeline_config

In [0]:
%run ../../utils/common_functions

In [0]:
silver = {name: spark.table(f"{SILVER_SCHEMA}.{name}") for name in DATASETS}
bronze = {name: spark.table(f"{BRONZE_SCHEMA}.{name}") for name in DATASETS}
dq = DataQualityChecker("silver")

# 1. Completeness and primary keys for every table
for name, df in silver.items():
    keys = DATASETS[name]["primary_key"]
    dq.not_empty(name, df)
    dq.not_null(name, df, keys)
    dq.unique(name, df, keys)

# 2. Every non-empty raw value must have parsed (a NULL here means the parsing rule missed a format)
parsed_columns = {
    "product": ["StandardCost"],
    "sales":   ["OrderDate", "Quantity", "UnitPrice", "Sales", "Cost"],
    "targets": ["TargetMonth", "Target"],
}
for name, cols in parsed_columns.items():
    for c in cols:
        unparsed = bronze[name].filter(F.trim(F.col(c)) != "").count() - silver[name].filter(F.col(c).isNotNull()).count()
        dq.check(name, f"{c} parsed for every raw value", unparsed)

# 3. Business rules
sales = silver["sales"]
dq.rule("sales", "Quantity > 0", sales, F.col("Quantity") <= 0)
dq.rule("sales", "Sales and Cost not negative", sales, (F.col("Sales") < 0) | (F.col("Cost") < 0))
dq.rule("sales", "OrderDate not in the future", sales, F.col("OrderDate") > F.current_date())
dq.rule("sales", "Sales = Quantity x UnitPrice (+/- 0.01)", sales,
        F.abs(F.col("Sales") - F.col("Quantity") * F.col("UnitPrice")) > 0.01, critical=False)
dq.rule("targets", "Target > 0", silver["targets"], F.col("Target") <= 0)
dq.rule("product", "StandardCost not negative", silver["product"], F.col("StandardCost") < 0)

# 4. Referential integrity
for parent, key in [("product", "ProductKey"), ("reseller", "ResellerKey"),
                    ("region", "SalesTerritoryKey"), ("salesperson", "EmployeeKey")]:
    dq.referential("sales", sales, parent, silver[parent], key)
dq.referential("targets", silver["targets"], "salesperson", silver["salesperson"], "EmployeeID")
dq.referential("salesperson_region", silver["salesperson_region"], "salesperson", silver["salesperson"], "EmployeeKey")
dq.referential("salesperson_region", silver["salesperson_region"], "region", silver["region"], "SalesTerritoryKey")

results = dq.results_df()
display(results)

In [0]:
write_delta_table(results, f"{SILVER_SCHEMA}.dq_check_results", f"{SILVER_PATH}_dq_check_results", mode="append")

# Raises if any critical check failed, which stops the job before gold is built
dq.assert_no_critical_failures()